# Filing Momentum ML Equity Curve

This notebook plots the Filing Momentum ML strategy equity curve against the S&P 500 proxy used by the backtest benchmark (`SPY`). It reads cached curve data from `outputs/filing_momentum_ml_equity_curve.csv` when available; set `REFRESH_BACKTEST = True` below to recompute it from the local raw data and manifest.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "outputs":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src" / "atlas_quant").exists():
    PROJECT_ROOT = Path("..").resolve()

sys.path.insert(0, str(PROJECT_ROOT / "src"))

OUTPUT_DIR = PROJECT_ROOT / "outputs"
CURVE_CSV = OUTPUT_DIR / "filing_momentum_ml_equity_curve.csv"
RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "filing_momentum_ml"
MANIFEST = PROJECT_ROOT / "data" / "manifests" / "filing_momentum_ml" / "data_manifest.json"
CHECKPOINT_ROOT = PROJECT_ROOT / "data" / "manifests" / "filing_momentum_ml"

START_QUARTER = "2015-03-31"
END_QUARTER = "2026-07-01"
BENCHMARK_SYMBOL = "SPY"
REFRESH_BACKTEST = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("project root:", PROJECT_ROOT)
print("curve cache:", CURVE_CSV)

In [ ]:
import pandas as pd

def run_backtest_equity_curve() -> pd.DataFrame:
    from argparse import Namespace

    from atlas_quant.cli.filing_momentum import (
        _build_run_inputs,
        _build_trading_calendar,
        _periods_from_args,
        load_normalized_bundle,
    )
    from atlas_quant.strategies.filing_momentum_ml.production.orchestration import (
        ProductionRunState,
        run_filing_momentum_production_backtest,
    )

    args = Namespace(
        raw_root=RAW_ROOT,
        manifest=MANIFEST,
        start_quarter=START_QUARTER,
        end_quarter=END_QUARTER,
        earnings_lag_days=None,
        benchmark=BENCHMARK_SYMBOL,
        checkpoint_root=CHECKPOINT_ROOT,
        dry_run=False,
    )
    bundle = load_normalized_bundle(args.raw_root)
    calendar = _build_trading_calendar(bundle)
    periods = _periods_from_args(args, required=True)
    inputs = _build_run_inputs(args, bundle, calendar, periods, with_report=False)
    result = run_filing_momentum_production_backtest(inputs)
    if result.state not in (ProductionRunState.COMPLETED, ProductionRunState.COMPLETED_WITH_WARNINGS):
        missing = ", ".join(d.name for d in result.missing_dependencies) or "none"
        raise RuntimeError(f"backtest did not complete: {result.state.value}; {result.blocked_reason}; missing={missing}")
    if result.backtest_result is None:
        raise RuntimeError("backtest completed without a BacktestResult")

    rows = [
        {
            "quarter_end": quarter_end.isoformat(),
            "filing_momentum_ml_equity": strategy_equity,
            "sp500_equity": benchmark_equity,
        }
        for quarter_end, strategy_equity, benchmark_equity in result.backtest_result.equity_curve()
    ]
    df = pd.DataFrame(rows)
    df.to_csv(CURVE_CSV, index=False)
    return df

if CURVE_CSV.exists() and not REFRESH_BACKTEST:
    curve = pd.read_csv(CURVE_CSV)
else:
    curve = run_backtest_equity_curve()

curve["quarter_end"] = pd.to_datetime(curve["quarter_end"])
curve = curve.sort_values("quarter_end").reset_index(drop=True)
curve.tail()

In [ ]:
from html import escape
from IPython.display import HTML, display

def render_equity_curve_svg(df, *, width=980, height=540):
    if df.empty:
        return "<p>No equity-curve data available.</p>"

    margin = {"top": 34, "right": 34, "bottom": 58, "left": 72}
    plot_w = width - margin["left"] - margin["right"]
    plot_h = height - margin["top"] - margin["bottom"]
    values = list(df["filing_momentum_ml_equity"]) + list(df["sp500_equity"])
    y_min = min(1.0, min(values))
    y_max = max(1.0, max(values))
    padding = (y_max - y_min) * 0.08 or 0.1
    y_min -= padding
    y_max += padding

    def x_at(i):
        if len(df) == 1:
            return margin["left"] + plot_w / 2
        return margin["left"] + (i / (len(df) - 1)) * plot_w

    def y_at(v):
        return margin["top"] + (y_max - v) / (y_max - y_min) * plot_h

    def polyline(column):
        return " ".join(f"{x_at(i):.1f},{y_at(v):.1f}" for i, v in enumerate(df[column]))

    grid = []
    labels = []
    for j in range(6):
        value = y_min + (y_max - y_min) * j / 5
        y = y_at(value)
        grid.append(f'<line x1="{margin["left"]}" y1="{y:.1f}" x2="{width - margin["right"]}" y2="{y:.1f}" stroke="#e5e7eb"/>')
        labels.append(f'<text x="{margin["left"] - 12}" y="{y + 4:.1f}" text-anchor="end" font-size="12" fill="#4b5563">{value:.2f}x</text>')

    tick_count = min(8, len(df))
    x_labels = []
    if tick_count:
        indexes = sorted(set(round(i * (len(df) - 1) / max(tick_count - 1, 1)) for i in range(tick_count)))
        for i in indexes:
            x = x_at(i)
            label = df.loc[i, "quarter_end"].strftime("%Y")
            x_labels.append(f'<text x="{x:.1f}" y="{height - 24}" text-anchor="middle" font-size="12" fill="#4b5563">{escape(label)}</text>')

    strategy_final = df["filing_momentum_ml_equity"].iloc[-1]
    sp500_final = df["sp500_equity"].iloc[-1]
    svg = f'''
    <svg viewBox="0 0 {width} {height}" width="100%" height="{height}" role="img" aria-label="Filing Momentum ML and S&P 500 equity curves">
      <rect width="{width}" height="{height}" fill="#ffffff"/>
      <text x="{margin['left']}" y="24" font-size="20" font-weight="700" fill="#111827">Filing Momentum ML vs S&amp;P 500 Equity Curve</text>
      {''.join(grid)}
      {''.join(labels)}
      <line x1="{margin['left']}" y1="{height - margin['bottom']}" x2="{width - margin['right']}" y2="{height - margin['bottom']}" stroke="#9ca3af"/>
      <line x1="{margin['left']}" y1="{margin['top']}" x2="{margin['left']}" y2="{height - margin['bottom']}" stroke="#9ca3af"/>
      {''.join(x_labels)}
      <polyline points="{polyline('sp500_equity')}" fill="none" stroke="#6b7280" stroke-width="3" stroke-linejoin="round" stroke-linecap="round"/>
      <polyline points="{polyline('filing_momentum_ml_equity')}" fill="none" stroke="#2563eb" stroke-width="3.5" stroke-linejoin="round" stroke-linecap="round"/>
      <rect x="{width - 315}" y="42" width="250" height="76" rx="6" fill="#ffffff" stroke="#d1d5db"/>
      <line x1="{width - 295}" y1="70" x2="{width - 255}" y2="70" stroke="#2563eb" stroke-width="4"/>
      <text x="{width - 245}" y="74" font-size="13" fill="#111827">Filing Momentum ML: {strategy_final:.2f}x</text>
      <line x1="{width - 295}" y1="98" x2="{width - 255}" y2="98" stroke="#6b7280" stroke-width="4"/>
      <text x="{width - 245}" y="102" font-size="13" fill="#111827">S&amp;P 500 (SPY): {sp500_final:.2f}x</text>
    </svg>
    '''
    return svg

display(HTML(render_equity_curve_svg(curve)))

In [ ]:
summary = pd.DataFrame(
    [
        {"series": "Filing Momentum ML", "final_equity": curve["filing_momentum_ml_equity"].iloc[-1], "total_return": curve["filing_momentum_ml_equity"].iloc[-1] - 1},
        {"series": "S&P 500 (SPY)", "final_equity": curve["sp500_equity"].iloc[-1], "total_return": curve["sp500_equity"].iloc[-1] - 1},
    ]
)
summary